In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date, timedelta
import os
import warnings
from tqdm.notebook import tqdm
from House2O import general_use

print(os.getcwd())

notebook_dir = os.getcwd()
sys.path.insert(0, os.path.join(os.getcwd(), "House2O"))
# test = notebook_dir.copy()

print(sys.path[0])

if 'House2O' in sys.modules:
    del sys.modules['House2O']

c:\Users\Matti\Documents\GitHub\House2O\House2O
c:\Users\Matti\Documents\GitHub\House2O\House2O\House2O


In [4]:
# ============================================================
# CONFIGURATION
# ============================================================
LAT           = 51.222
LON           = 4.401
START_DATE    = date(2024, 3, 21)
END_DATE      = date(2024, 6, 21)
HOURS         = list(range(6, 22, 1))
DAY_STEP      = 1
ANGLE         = 38
AZIMUTH       = 105
LABEL         = "SpringDataOptimized"
SAVE_DIR      = r"C:\Users\Matti\Documents\GitHub\House2O\Thermal_sim\AttemptingSomething"

# ============================================================
CACHE_FILE = os.path.join(SAVE_DIR, f"{LABEL.replace(' ', '_').replace('/', '-')}_cache.csv")
os.makedirs(SAVE_DIR, exist_ok=True)

HOUR_STEP_SIZE = HOURS[1] - HOURS[0]

# ── Build day list ────────────────────────────────────────────
all_days, d = [], START_DATE
while d <= END_DATE:
    all_days.append(d)
    d += timedelta(days=DAY_STEP)

total_calls = len(all_days) * len(HOURS)

print(f"Days:       {START_DATE} → {END_DATE} (every {DAY_STEP} days)")
print(f"Hours:      {HOURS[0]:02d}:00 → {HOURS[-1]:02d}:00, step: {HOUR_STEP_SIZE}h")
print(f"Angle:      {ANGLE}°")
print(f"Azimuth:    {AZIMUTH}°")
print(f"Label:      {LABEL}")

# ── Load or init cache ────────────────────────────────────────
if os.path.exists(CACHE_FILE):
    cache_df = pd.read_csv(CACHE_FILE)
    cache_df = cache_df.drop_duplicates(subset=["date", "hour"], keep="last")
    cache    = set(zip(cache_df["date"], cache_df["hour"]))
    records  = cache_df.to_dict("records")
    print(f"Resuming from cache — {len(records)} results already stored.")
else:
    cache, records = set(), []
    print("Starting fresh.")

total_remaining = max(0, total_calls - len(cache))
print(f"Total computations: {total_remaining:,}\n")

# ── Main sweep ────────────────────────────────────────────────
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)

    bar = tqdm(total=total_remaining, desc="Processing", unit="call", dynamic_ncols=True)

    for day in all_days:
        day_str = day.strftime("%Y-%m-%d")
        for hour in HOURS:
            key = (day_str, hour)
            if key in cache:
                continue

            datetime_str = f"{day_str} {hour:02d}:00"
            try:
                _, _, power, power_glass = general_use(
                    LAT=LAT, LON=LON,
                    DATETIME=datetime_str,
                    surface_tilt=ANGLE,
                    surface_azimuth=AZIMUTH,
                    print_details=False,
                )
            except Exception as e:
                tqdm.write(f"  [ERROR] {datetime_str}: {e}")
                power       = 0.0
                power_glass = 0.0

            records.append({
                "date":        day_str,
                "hour":        hour,
                "power":       power,
                "power_glass": power_glass,
            })
            cache.add(key)
            bar.update(1)

        # save after each day so progress is never lost on a crash
        pd.DataFrame(records).to_csv(CACHE_FILE, index=False)

    bar.close()

# ── Aggregate to daily totals ─────────────────────────────────
df = pd.DataFrame(records).drop_duplicates(subset=["date", "hour"], keep="last")

def trapezoid_energy(group):
    g = group.sort_values("hour").dropna(subset=["power", "power_glass"])
    return pd.Series({
        "energy_Wh_m2":       np.trapezoid(g["power"],       dx=HOUR_STEP_SIZE),
        "energy_glass_Wh_m2": np.trapezoid(g["power_glass"], dx=HOUR_STEP_SIZE),
    })

daily = df.groupby("date").apply(trapezoid_energy, include_groups=False).reset_index()

total_water = daily["energy_Wh_m2"].sum()
total_glass = daily["energy_glass_Wh_m2"].sum()

print(f"\n{'='*55}")
print(f"  LABEL:              {LABEL}")
print(f"  ANGLE:              {ANGLE}°  |  AZIMUTH: {AZIMUTH}°")
print(f"  Total water energy: {total_water:.1f} Wh/m²")
print(f"  Total glass loss:   {total_glass:.1f} Wh/m²")
print(f"{'='*55}")

# ── Save final aggregated CSV ─────────────────────────────────
label_slug   = LABEL.replace(" ", "_").replace("/", "-")
# out_file     = os.path.join(SAVE_DIR, f"{label_slug}_daily_energy.csv")
# daily.to_csv(out_file, index=False)
# print(f"Saved: {out_file}")

Days:       2024-03-21 → 2024-06-21 (every 1 days)
Hours:      06:00 → 21:00, step: 1h
Angle:      38°
Azimuth:    105°
Label:      SpringDataOptimized
Starting fresh.
Total computations: 1,488



Processing:   0%|          | 0/1488 [00:00<?, ?call/s]


  LABEL:              SpringDataOptimized
  ANGLE:              38°  |  AZIMUTH: 105°
  Total water energy: 158067.4 Wh/m²
  Total glass loss:   24189.3 Wh/m²
